# Stage 27g TIME_SPEC 100MHz Production Control + Preview

Production-only control for TIME/SPEC science streams and 8-lane DAC-ADC loopback, with RF-reconstructed waveform and spectrum preview only.


In [ ]:
from pathlib import Path
import asyncio
import importlib
import json
import sys
import time
import urllib.request

import ipywidgets as W
import numpy as np
import plotly.graph_objects as go
from IPython.display import HTML, display

candidate_roots = [
    Path.cwd().resolve(),
    Path.cwd().resolve().parent,
    Path('/home/xilinx/jupyter_notebooks/t510_fengine'),
    Path('/home/xilinx/t510_fengine_bringup'),
    Path('/home/astrolab/demo-ant'),
]
PROJECT_ROOT = None
for root in candidate_roots:
    if (root / 'overlay' / 't510_fengine.bit').exists() and (root / 'python' / 't510_fengine.py').exists():
        PROJECT_ROOT = root
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('overlay/t510_fengine.bit and python/t510_fengine.py were not found')

sys.path.insert(0, str(PROJECT_ROOT))
import python.t510_clock as t510_clock_module
import python.t510_fengine as t510_fengine_module
importlib.reload(t510_clock_module)
importlib.reload(t510_fengine_module)

T510FEngine = t510_fengine_module.T510FEngine
ObservationSpectrumStabilizer = t510_fengine_module.ObservationSpectrumStabilizer

BITFILE = PROJECT_ROOT / 'overlay' / 't510_fengine.bit'
EXPECTED_CORE_VERSION = 0x00010025
COLORS = ['#0b5cad', '#c45200', '#217a3b', '#b3261e', '#6f42c1', '#5f6368', '#008b8b', '#9a7b00']

state = {
    'core': None,
    'live': False,
    'task': None,
    'last_error': None,
    'last_cfg': None,
    'last_preview': None,
    'last_rates': None,
    'last_rates_time': 0.0,
    'stabilizer': ObservationSpectrumStabilizer(),
}


def set_status(text, *, kind='info'):
    color = {'ok': '#0b7a3b', 'warn': '#9a6a00', 'error': '#b3261e', 'info': '#354052'}.get(kind, '#354052')
    status_html.value = f"<pre style='margin:0;color:{color}'>{text}</pre>"


def core():
    if state['core'] is None:
        raise RuntimeError('Click Load first')
    return state['core']


def visible_mask():
    return 0xFF


def phase_values():
    return [float(widget.value) for widget in phase_ch]


def amplitude_code():
    return int(round(float(amplitude_percent.value) / 100.0 * 8192.0))


def dac_signal_hz():
    return float(dac_mhz.value) * 1_000_000.0


def expected_signal_hz():
    return dac_signal_hz()


def science_output_mode_value():
    return str(science_output_mode.value)


def capture_input_mask():
    return visible_mask()


def dac_enable_mask():
    return visible_mask()


def adc_active_mask():
    return T510FEngine.complex_input_mask_to_adc_active_mask(visible_mask())


def sync_rust_receiver_display():
    url = str(rust_receiver_url.value).strip().rstrip('/')
    if not url:
        return {'skipped': True, 'reason': 'empty Rust receiver URL'}
    payload = {
        'bandwidth_mhz': int(bandwidth_mhz.value),
        'center_mhz': float(center_mhz.value),
        'expected_mhz': float(dac_mhz.value),
        'dac_mhz': float(dac_mhz.value),
        'phase_deg_by_channel': phase_values(),
        'channel_mask': visible_mask(),
        'time_window_us': float(time_window_us.value),
        'display_points': int(scope_display_points.value),
        'vertical_scale': float(scope_y_codes.value),
        'paused': False,
    }
    req = urllib.request.Request(
        url + '/api/config',
        data=json.dumps(payload).encode('utf-8'),
        headers={'Content-Type': 'application/json'},
        method='POST',
    )
    try:
        with urllib.request.urlopen(req, timeout=2.0) as response:
            response.read()
        return {'ok': True, 'url': url, 'payload': payload}
    except Exception as exc:
        return {'ok': False, 'url': url, 'error': f'{type(exc).__name__}: {exc}', 'payload': payload}


def fetch_rust_state():
    url = str(rust_receiver_url.value).strip().rstrip('/')
    if not url:
        return {'skipped': True, 'reason': 'empty Rust receiver URL'}
    try:
        with urllib.request.urlopen(url + '/api/state', timeout=2.0) as response:
            return json.loads(response.read().decode('utf-8'))
    except Exception as exc:
        return {'ok': False, 'url': url, 'error': f'{type(exc).__name__}: {exc}'}


def capture_count():
    return T510FEngine.observation_capture_count(
        sample_rate_hz=245_760_000.0,
        time_window_us=float(time_window_us.value),
        oversample=float(preview_oversample.value),
        min_count=512,
        max_count=int(preview_max_samples.value),
    )


def compact_status(status):
    keys = [
        'core_version', 'mode_name', 'science_output_mode', 'sample_rate_hz',
        'time_packet_count', 'spec_packet_count', 'time_dropped_count', 'spec_dropped_count',
        'pfb_overflow_count', 'pfb_xfft_event_count', 'pfb_xfft_tlast_missing_count',
        'pfb_xfft_tlast_unexpected_count', 'pfb_xfft_fft_overflow_count',
        'pfb_xfft_data_out_halt_count', 'pfb_capture_backpressure_count',
        'science_dropped_beat_count', 'tx_route_miss_count', 'tx_route_error_count',
        'tx_frame_dropped_count', 'tx_frame_built_count', 'cmac_link_up', 'cmac_tx_ready', 'rfdc_current_valid_mask',
    ]
    return {key: status.get(key) for key in keys if key in status}


def update_status_panel(status=None, analysis=None, rates=None, elapsed=None):
    c = state.get('core')
    if c is None:
        board_html.value = '<pre style="margin:0">Board not loaded.</pre>'
        rust_html.value = '<pre style="margin:0">Rust receiver not queried.</pre>'
        preview_html.value = '<pre style="margin:0">Preview not captured.</pre>'
        return
    if rates is None:
        rates = get_rates(c, force=True)
    status = rates['status'] if status is None else status
    science = rates.get('science', {})
    tx = rates.get('tx', {})
    channelizer = rates.get('channelizer', {})
    time_routes = rates.get('time_routes', [])
    spec_routes = rates.get('spec_routes', [])
    time_hit_routes = sum(1 for route in time_routes if int(route.get('enable', 0)) and int(route.get('hit_count', 0)) > 0)
    spec_hit_routes = sum(1 for route in spec_routes if int(route.get('enable', 0)) and int(route.get('hit_count', 0)) > 0)
    board_html.value = (
        '<pre style="margin:0">'
        f"core=0x{int(status.get('core_version', 0)):08x} mode={status.get('mode_name', status.get('mode'))} science={science.get('science_output_mode')}\n"
        f"TIME packets={int(status.get('time_packet_count', 0))} drops={int(status.get('time_dropped_count', 0))} bytes={int(status.get('time_udp_byte_count', 0))} route={time_hit_routes}/{len(time_routes)}\n"
        f"SPEC packets={int(status.get('spec_packet_count', 0))} drops={int(status.get('spec_dropped_count', 0))} bytes={int(status.get('spec_udp_byte_count', 0))} route={spec_hit_routes}/{len(spec_routes)}\n"
        f"PFB overflow={int(status.get('pfb_overflow_count', 0))} xfft_event={int(status.get('pfb_xfft_event_count', 0))} science_drop={int(status.get('science_dropped_beat_count', 0))} peak_chan={channelizer.get('peak_chan', status.get('pfb_peak_chan'))}\n"
        f"TX frames={int(status.get('tx_frame_built_count', 0))} route_miss={int(status.get('tx_route_miss_count', 0))} route_error={int(status.get('tx_route_error_count', 0))} frame_drop={int(status.get('tx_frame_dropped_count', 0))}\n"
        f"CMAC link={tx.get('link_up', status.get('cmac_link_up'))} tx_ready={tx.get('cmac_tx_ready', status.get('cmac_tx_ready'))} local_fault={tx.get('tx_local_fault')} remote_fault={tx.get('tx_remote_fault')}\n"
        f"SPEC route_incomplete={int(science.get('spec_route_incomplete', 0))} RFDC active=0x{int(status.get('rfdc_active_mask', 0)):04x} valid=0x{int(status.get('rfdc_current_valid_mask', 0)):04x} seen=0x{int(status.get('rfdc_seen_valid_mask', 0)):04x}"
        '</pre>'
    )
    rust = rates.get('rust', {})
    if isinstance(rust, dict) and 'stats' in rust:
        s = rust['stats']
        rust_html.value = (
            '<pre style="margin:0">'
            f"url={str(rust_receiver_url.value).strip().rstrip('/')} backend={s.get('backend')} fanout={s.get('fanout_mode')} workers={s.get('active_worker_count')}/{s.get('worker_count')}\n"
            f"selected={s.get('selected_bandwidth_mhz')}MHz detected={s.get('detected_bandwidth_mhz')}MHz mismatch={s.get('selected_detected_mismatch')}\n"
            f"TIME packets={s.get('time_packets')} processed={float(s.get('rx_processed_gbps', 0.0)):.3f}Gbps gaps={s.get('seq_gaps')}/{s.get('frame_gaps')}/{s.get('sample0_gaps')}\n"
            f"SPEC packets={s.get('spec_packets')} processed={float(s.get('spec_processed_gbps', 0.0)):.3f}Gbps gaps={s.get('spec_seq_gaps')}/{s.get('spec_frame_gaps')}\n"
            f"drops parse={s.get('parse_errors')} ring={s.get('ring_drops')} worker_ring={s.get('worker_ring_drops')} kernel={s.get('kernel_drops')}\n"
            f"preview waveform={float(s.get('display_update_hz', 0.0)):.2f}Hz spectrum={float(s.get('spectrum_update_hz', 0.0)):.2f}Hz ws={s.get('websocket_clients')}"
            '</pre>'
        )
    else:
        rust_html.value = '<pre style="margin:0">' + json.dumps(rust, indent=2, sort_keys=True, default=str) + '</pre>'
    if analysis is not None:
        peaks = analysis.get('peaks', {})
        lines = []
        for ch in range(8):
            if ch in peaks:
                pk = peaks[ch]
                lines.append(f"CH{ch} RF={float(pk.get('rf_peak_mhz', 0.0)):.4f}MHz peak={float(pk.get('peak_dbfs', 0.0)):.2f}dBFS SNR={float(pk.get('snr_db', 0.0)):.1f}dB")
        preview_html.value = '<pre style="margin:0">' + '\n'.join(lines[:8]) + '</pre>'


def load_core(_=None):
    try:
        state['core'] = T510FEngine(str(BITFILE), download=bool(download_bitstream.value))
        status = state['core'].read_status()
        core_version = int(status.get('core_version', 0))
        kind = 'ok' if core_version == EXPECTED_CORE_VERSION else 'warn'
        set_status(
            f"Loaded {BITFILE}\ncore_version=0x{core_version:08x} expected=0x{EXPECTED_CORE_VERSION:08x}\n" +
            json.dumps(compact_status(status), indent=2, sort_keys=True),
            kind=kind,
        )
        update_status_panel(status=status)
    except Exception as exc:
        state['last_error'] = exc
        set_status(f'Load failed: {type(exc).__name__}: {exc}', kind='error')


def apply_production_config(*, initialize=False, start=False):
    c = core()
    if science_output_mode_value() == 'time_spec' and int(bandwidth_mhz.value) == 200:
        raise ValueError('Stage 27g production rejects TIME_SPEC at 200 MHz')
    observation = c.apply_mts_locked_observation_config(
        observe_center_hz=float(center_mhz.value) * 1_000_000.0,
        dac_signal_hz=dac_signal_hz(),
        expected_signal_hz=expected_signal_hz(),
        view_bw_hz=float(bandwidth_mhz.value) * 1_000_000.0,
        amplitude=amplitude_code(),
        phase_deg=0.0,
        phase_deg_by_channel=phase_values(),
        enable_mask=dac_enable_mask(),
        adc_active_mask=adc_active_mask(),
        initialize=initialize,
        start=False,
        require_full_clock_lock=True,
        require_mts=True,
        force_clock_reconfigure=bool(initialize),
        input_source_mode='dac_loopback',
        clock_ref='tcxo_10mhz',
        sync_mode='free_run',
    )
    science = c.configure_science_27g(
        bandwidth_mhz=int(bandwidth_mhz.value),
        output_mode=science_output_mode_value(),
        dst_ip=str(dst_ip.value).strip(),
        dst_mac=str(dst_mac.value).strip(),
        src_ip=str(src_ip.value).strip(),
        src_mac=str(src_mac.value).strip(),
        time_dst_port_base=int(time_dst_port_base.value),
        spec_dst_port_base=int(spec_dst_port_base.value),
        time_src_port_base=int(time_src_port_base.value),
        spec_src_port_base=int(spec_src_port_base.value),
        input_mask=0x00FF,
        clear_counters=True,
        start=start,
    )
    rust_sync = sync_rust_receiver_display()
    cfg = {'observation': observation, 'science': science, 'rust_receiver_sync': rust_sync}
    state['last_cfg'] = cfg
    time.sleep(0.2)
    update_status_panel()
    return cfg


def apply_clicked(_=None):
    try:
        cfg = apply_production_config(initialize=False, start=True)
        set_status(
            'Applied Stage 27g production config\n' +
            f"mode={science_output_mode_value().upper()} bandwidth={int(bandwidth_mhz.value)}MHz center={float(center_mhz.value):.3f}MHz dac={float(dac_mhz.value):.3f}MHz\n" +
            json.dumps({'science': cfg['science'], 'rust_receiver_sync': cfg['rust_receiver_sync']}, indent=2, sort_keys=True, default=str),
            kind='ok',
        )
    except Exception as exc:
        state['last_error'] = exc
        set_status(f'Apply failed: {type(exc).__name__}: {exc}', kind='error')


def init_clicked(_=None):
    try:
        cfg = apply_production_config(initialize=True, start=True)
        set_status(
            'Initialized RFDC/DAC loopback and started Stage 27g science stream\n' +
            json.dumps({'science': cfg['science'], 'rust_receiver_sync': cfg['rust_receiver_sync']}, indent=2, sort_keys=True, default=str),
            kind='ok',
        )
    except Exception as exc:
        state['last_error'] = exc
        set_status(f'Init failed: {type(exc).__name__}: {exc}', kind='error')


def stop_clicked(_=None):
    try:
        state['live'] = False
        task = state.get('task')
        if task is not None:
            task.cancel()
        core().stop()
        set_status('Stopped production stream and live preview.', kind='warn')
        update_status_panel()
    except Exception as exc:
        state['last_error'] = exc
        set_status(f'Stop failed: {type(exc).__name__}: {exc}', kind='error')


def reduce_xy(x, y, max_points):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if x.size <= max_points:
        return x, y
    idx = np.linspace(0, x.size - 1, int(max_points)).astype(int)
    return x[idx], y[idx]


def update_waveform_plot(analysis):
    with scope_fig.batch_update():
        for ch in range(8):
            trace = scope_fig.data[ch]
            if ch not in analysis['scope']:
                trace.visible = False
                continue
            scope = analysis['scope'][ch]
            x, y = reduce_xy(scope['time_us'], scope['rf_equivalent_waveform'], int(scope_display_points.value))
            trace.x = x
            trace.y = y
            trace.visible = True
            trace.name = f'CH{ch}'
        scope_fig.layout.title = 'RF reconstructed waveform from real RFDC preview IQ'
        scope_fig.layout.yaxis.range = [-float(scope_y_codes.value), float(scope_y_codes.value)]


def update_spectrum_plot(analysis):
    xmin = float(center_mhz.value) - float(bandwidth_mhz.value) / 2.0
    xmax = float(center_mhz.value) + float(bandwidth_mhz.value) / 2.0
    with spectrum_fig.batch_update():
        for ch in range(8):
            trace = spectrum_fig.data[ch]
            if ch not in analysis['spectrum']:
                trace.visible = False
                continue
            spectrum = analysis['spectrum'][ch]
            peak = analysis['peaks'].get(ch, {})
            update = state['stabilizer'].update_channel(
                ch,
                spectrum,
                peak,
                smoothing_enabled=bool(smoothing_enabled.value),
                alpha=float(smoothing_alpha.value),
            )
            x, y = reduce_xy(update['rf_mhz'], update['display_power_dbfs'], int(spectrum_display_points.value))
            trace.x = x
            trace.y = y
            trace.visible = True
            trace.name = f'CH{ch}'
        spectrum_fig.layout.xaxis.range = [xmin, xmax]
        spectrum_fig.layout.yaxis.range = [int(spectrum_floor_db.value), int(spectrum_top_db.value)]
        spectrum_fig.layout.title = f'Production RF spectrum center={float(center_mhz.value):.1f} MHz BW={float(bandwidth_mhz.value):.0f} MHz'


def get_rates(c, *, force=False):
    now = time.monotonic()
    if not force and state.get('last_rates') is not None and now - state.get('last_rates_time', 0.0) < 0.8:
        return state['last_rates']
    status = c.read_status()
    science = c.read_science_output_status()
    tx = c.read_tx_status()
    channelizer = c.read_channelizer_status()
    time_routes = c.read_time_route_table()
    spec_routes = c.read_spec_route_table()
    rust = fetch_rust_state()
    rates = {'status': status, 'science': science, 'tx': tx, 'channelizer': channelizer, 'time_routes': time_routes, 'spec_routes': spec_routes, 'rust': rust}
    state['last_rates'] = rates
    state['last_rates_time'] = now
    return rates


def capture_once(_=None):
    try:
        c = core()
        t0 = time.monotonic()
        preview = c.capture_preview_fast(n=capture_count(), input_mask=capture_input_mask(), timeout=float(timeout_s.value))
        state['last_preview'] = preview
        analysis = c.compute_observation_view(
            preview,
            observe_center_hz=float(center_mhz.value) * 1_000_000.0,
            view_bw_hz=float(bandwidth_mhz.value) * 1_000_000.0,
            dac_signal_hz=dac_signal_hz(),
            expected_signal_hz=expected_signal_hz(),
            time_window_us=float(time_window_us.value),
            oversample=float(preview_oversample.value),
            phase_ref_input=0,
            stabilize_phase=False,
            display_phase_deg=0.0,
            phase_deg_by_channel=phase_values(),
            input_source_mode='dac_loopback',
        )
        update_waveform_plot(analysis)
        update_spectrum_plot(analysis)
        rates = get_rates(c, force=True)
        elapsed = time.monotonic() - t0
        update_status_panel(analysis=analysis, rates=rates, elapsed=elapsed)
        set_status(f"Captured production preview sample0={int(preview['sample0'])} count={int(preview['count'])} elapsed={elapsed*1000.0:.1f}ms", kind='ok')
    except Exception as exc:
        state['last_error'] = exc
        set_status(f'Capture failed: {type(exc).__name__}: {exc}', kind='error')


async def live_loop():
    while state.get('live', False):
        capture_once()
        await asyncio.sleep(max(1.0 / max(float(refresh_hz.value), 0.1), float(min_ui_sleep_ms.value) / 1000.0))


def start_live(_=None):
    try:
        if state.get('live', False):
            return
        state['live'] = True
        state['task'] = asyncio.create_task(live_loop())
        set_status('Live production preview started.', kind='ok')
    except Exception as exc:
        state['last_error'] = exc
        set_status(f'Start live failed: {type(exc).__name__}: {exc}', kind='error')


def stop_live(_=None):
    state['live'] = False
    task = state.get('task')
    if task is not None:
        task.cancel()
    set_status('Live preview stopped.', kind='warn')


def validate_board(_=None):
    try:
        result = core().run_stage27g_time_spec_convergence_validation(
            configure=False,
            expected_core_version=EXPECTED_CORE_VERSION,
            bandwidth_mhz=int(bandwidth_mhz.value),
            output_mode=science_output_mode_value(),
            seconds=float(validation_seconds.value),
            min_packet_delta=1,
        )
        update_status_panel()
        set_status(json.dumps(result, indent=2, sort_keys=True, default=str), kind='ok' if result.get('ok') else 'error')
    except Exception as exc:
        state['last_error'] = exc
        set_status(f'Board validation failed: {type(exc).__name__}: {exc}', kind='error')


def on_mode_or_bw_change(change=None):
    if science_output_mode_value() == 'time_spec' and int(bandwidth_mhz.value) == 200:
        set_status('TIME_SPEC 200MHz is rejected by Stage 27g production; choose 100MHz or change mode.', kind='warn')


def apply_phase_only(_=None):
    try:
        c = core()
        c.configure_dac_tone_bank(
            freq_hz=dac_signal_hz(),
            amplitude=amplitude_code(),
            phase_offset_deg=0.0,
            phase_deg_by_channel=phase_values(),
            enable_mask=dac_enable_mask(),
            mode='single_tone',
        )
        epoch = c.reset_dac_phase()
        sync_rust_receiver_display()
        set_status(f'Applied DAC phases epoch={epoch}.', kind='ok')
    except Exception as exc:
        state['last_error'] = exc
        set_status(f'Phase apply failed: {type(exc).__name__}: {exc}', kind='error')


style = {'description_width': '130px'}
wide = W.Layout(width='430px')
download_bitstream = W.Checkbox(value=True, description='Download bitstream')

rust_receiver_url = W.Text(value='http://192.168.100.192:8089', description='Rust receiver', style=style, layout=wide)
dst_ip = W.Text(value='10.0.1.16', description='Receiver IP', style=style, layout=wide)
dst_mac = W.Text(value='08:c0:eb:d5:95:b2', description='Receiver MAC', style=style, layout=wide)
src_ip = W.Text(value='10.0.1.1', description='Source IP', style=style, layout=wide)
src_mac = W.Text(value='02:00:00:00:00:01', description='Source MAC', style=style, layout=wide)
time_dst_port_base = W.IntText(value=4300, description='TIME dst port', style=style, layout=wide)
spec_dst_port_base = W.IntText(value=4308, description='SPEC dst port', style=style, layout=wide)
time_src_port_base = W.IntText(value=4000, description='TIME src port', style=style, layout=wide)
spec_src_port_base = W.IntText(value=4008, description='SPEC src port', style=style, layout=wide)

science_output_mode = W.Dropdown(options=[('TIME + SPEC', 'time_spec'), ('TIME only', 'time_only'), ('SPEC only', 'spec_only')], value='time_spec', description='Mode', style=style, layout=wide)
bandwidth_mhz = W.Dropdown(options=[('100 MHz', 100), ('20 MHz', 20), ('200 MHz', 200)], value=100, description='Bandwidth', style=style, layout=wide)
center_mhz = W.FloatText(value=200.0, description='Center MHz', style=style, layout=wide)
dac_mhz = W.FloatText(value=200.0, description='DAC RF MHz', style=style, layout=wide)
amplitude_percent = W.FloatSlider(value=25.0, min=0.0, max=100.0, step=1.0, description='DAC amplitude %', readout_format='.0f', style=style, layout=wide)
phase_ch = [W.FloatSlider(value=0.0, min=-180.0, max=180.0, step=1.0, description=f'CH{i}', readout_format='.0f', continuous_update=False, style={'description_width': '48px'}, layout=W.Layout(width='260px')) for i in range(8)]

scope_y_codes = W.IntSlider(value=4096, min=256, max=32768, step=256, description='Y scale codes', style=style, layout=wide)
scope_display_points = W.IntSlider(value=1024, min=256, max=4096, step=256, description='Scope points', style=style, layout=wide)
time_window_us = W.FloatSlider(value=0.25, min=0.05, max=2.0, step=0.05, description='Window us', readout_format='.2f', style=style, layout=wide)
preview_oversample = W.FloatSlider(value=2.5, min=1.0, max=8.0, step=0.5, description='Preview oversample', style=style, layout=wide)
preview_max_samples = W.Dropdown(options=[512, 1024, 2048, 4096], value=1024, description='Max samples', style=style, layout=wide)
refresh_hz = W.FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1, description='Preview Hz', readout_format='.1f', style=style, layout=wide)
min_ui_sleep_ms = W.IntSlider(value=30, min=0, max=250, step=10, description='UI sleep ms', style=style, layout=wide)
timeout_s = W.FloatSlider(value=1.0, min=0.2, max=5.0, step=0.2, description='Timeout s', readout_format='.1f', style=style, layout=wide)
validation_seconds = W.FloatSlider(value=2.0, min=0.5, max=30.0, step=0.5, description='Validate s', readout_format='.1f', style=style, layout=wide)
spectrum_floor_db = W.IntSlider(value=-120, min=-160, max=-20, step=5, description='Spectrum min', style=style, layout=wide)
spectrum_top_db = W.IntSlider(value=-20, min=-80, max=10, step=5, description='Spectrum max', style=style, layout=wide)
spectrum_display_points = W.IntSlider(value=512, min=128, max=2048, step=128, description='Spectrum points', style=style, layout=wide)
smoothing_enabled = W.Checkbox(value=True, description='Smooth spectrum')
smoothing_alpha = W.FloatSlider(value=0.25, min=0.05, max=1.0, step=0.05, description='Smooth alpha', style=style, layout=wide)

load_btn = W.Button(description='Load', icon='download', button_style='primary', layout=W.Layout(width='100px'))
init_btn = W.Button(description='Init + Start', icon='play', button_style='success', layout=W.Layout(width='130px'))
apply_btn = W.Button(description='Apply', icon='check', button_style='success', layout=W.Layout(width='100px'))
phase_btn = W.Button(description='Apply phase', icon='sliders', layout=W.Layout(width='130px'))
capture_btn = W.Button(description='Capture', icon='camera', layout=W.Layout(width='110px'))
start_live_btn = W.Button(description='Live preview', icon='play', layout=W.Layout(width='130px'))
stop_live_btn = W.Button(description='Stop live', icon='stop', button_style='warning', layout=W.Layout(width='110px'))
validate_btn = W.Button(description='Board gate', icon='shield-check', button_style='warning', layout=W.Layout(width='120px'))
stop_btn = W.Button(description='Stop stream', icon='square', button_style='warning', layout=W.Layout(width='120px'))

status_html = W.HTML(value='<pre style="margin:0">Load the overlay to begin.</pre>')
board_html = W.HTML(value='<pre style="margin:0">Board not loaded.</pre>')
rust_html = W.HTML(value='<pre style="margin:0">Rust receiver not queried.</pre>')
preview_html = W.HTML(value='<pre style="margin:0">Capture a frame to populate preview metrics.</pre>')

scope_fig = go.FigureWidget()
for ch in range(8):
    scope_fig.add_trace(go.Scattergl(x=[], y=[], mode='lines', name=f'CH{ch}', line={'color': COLORS[ch], 'width': 2 if ch == 0 else 1.2}, visible=(ch == 0)))
scope_fig.update_layout(height=360, margin={'l': 58, 'r': 20, 't': 38, 'b': 46}, xaxis_title='preview time (us)', yaxis_title='ADC code', template='plotly_white', showlegend=True, title='RF reconstructed waveform from real RFDC preview IQ')

spectrum_fig = go.FigureWidget()
for ch in range(8):
    spectrum_fig.add_trace(go.Scattergl(x=[], y=[], mode='lines', name=f'CH{ch}', line={'color': COLORS[ch], 'width': 2 if ch == 0 else 1.2}, visible=(ch == 0)))
spectrum_fig.update_layout(height=360, margin={'l': 58, 'r': 20, 't': 38, 'b': 46}, xaxis_title='RF frequency (MHz)', yaxis_title='power (dBFS)', template='plotly_white', showlegend=True, title='Production RF spectrum')

load_btn.on_click(load_core)
init_btn.on_click(init_clicked)
apply_btn.on_click(apply_clicked)
phase_btn.on_click(apply_phase_only)
capture_btn.on_click(capture_once)
start_live_btn.on_click(start_live)
stop_live_btn.on_click(stop_live)
validate_btn.on_click(validate_board)
stop_btn.on_click(stop_clicked)
science_output_mode.observe(on_mode_or_bw_change, names='value')
bandwidth_mhz.observe(on_mode_or_bw_change, names='value')

network_box = W.VBox([dst_ip, dst_mac, src_ip, src_mac, time_dst_port_base, spec_dst_port_base, time_src_port_base, spec_src_port_base, rust_receiver_url])
mode_box = W.VBox([science_output_mode, bandwidth_mhz, center_mhz, dac_mhz, amplitude_percent])
phase_grid = W.GridBox(phase_ch, layout=W.Layout(grid_template_columns='repeat(2, 270px)', grid_gap='4px 8px'))
phase_box = W.VBox([phase_grid, phase_btn])
preview_box = W.VBox([scope_y_codes, scope_display_points, time_window_us, preview_oversample, preview_max_samples, refresh_hz, min_ui_sleep_ms, timeout_s, validation_seconds, spectrum_floor_db, spectrum_top_db, spectrum_display_points, smoothing_enabled, smoothing_alpha])
controls = W.Tab(children=[network_box, mode_box, phase_box, preview_box])
for idx, title in enumerate(['Network', 'Science + DAC', '8-lane phase', 'Preview']):
    controls.set_title(idx, title)

display(HTML('<h2>Stage 27g TIME_SPEC 100MHz Production Control + Preview</h2>'))
display(W.HBox([load_btn, init_btn, apply_btn, capture_btn, start_live_btn, stop_live_btn, validate_btn, stop_btn]))
display(controls)
display(status_html)
display(W.HBox([scope_fig, spectrum_fig]))
display(W.Tab(children=[board_html, rust_html, preview_html]))
